# SLMStack client and viewer example

Start the SLMStack server from a terminal first, then run these notebook cells to connect a client, open the shared-memory viewer, and send the example masks.

python -m JazLabs.hardware.SLM.SLMStack.SLM_Server --slm-type "Blink OverDrive Plus"

In [1]:
# Python libs
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np

plt.style.use('dark_background')

# Let the notebook run from a source checkout without requiring pip install -e .
repo_root = Path.cwd()
while repo_root.name != 'JazLabs' and repo_root != repo_root.parent:
    repo_root = repo_root.parent
src_path = repo_root / 'src'
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import JazLabs.utils.GenerateSimplePhaseMasks as slm_masks
import JazLabs.hardware.SLM.PhaseMaskClass as PhaseMaskClass
from JazLabs.hardware.SLM.SLMStack import SLMClient, SLMOutputViewer

In [2]:
%load_ext autoreload
%aimport JazLabs.hardware.SLM.SLMStack.SLM_Client
%aimport JazLabs.hardware.SLM.SLMStack.SLM_Viewer
%aimport JazLabs.hardware.SLM.PhaseMaskClass
%aimport JazLabs.utils.GenerateSimplePhaseMasks
%autoreload 1

## Connect to the command-line server

The server owns the physical SLM and publishes a local shared-memory buffer for the viewer. The notebook only starts a client and viewer.

In [3]:
slmobj_client = SLMClient(
    host='127.0.0.1',
    command_port=5555,
    display_pub_port=5556,
    timeout_ms=5000,
    attach_viewer_shared_memory=False,
)

# props = slmobj_client.GetProperties()
# print(props)

# viewer = SLMOutputViewer(
#     shm_name=props['viewer_shared_memory_name'],
#     shape=props['viewer_shape'],
#     dtype=props['viewer_dtype'],
#     zoom=0.2,
#     fps=30,
# )
# viewer.startProcess()

In [4]:
phasemaskobj = PhaseMaskClass.PhaseMaskObject(
    SLMObject=slmobj_client,
    ActiveRGBChannels=['Red'],
    pixel_size=17e-6,
    wavelength=1550e-9,
)


        Zern Coefs:
        0:  piston    ( 0,  0 )
        1:  Tiltx     (1,  -1 )
        2:  Tilty     ( 1,  1 )
        3:  Astigx    (2,  -2 )
        4:  Defocus   ( 2,  0 )
        5:  Astigy    ( 2,  2 )
        6:  Trefilx   (3,  -3 )
        7:  Comax     (3,  -1 )
        8:  Comay     ( 1,  3 )
        9:  Trefoily  ( 3,  3 )
        12:  Spherical ( 4,  0 )
        
1024
1024
[  1   1 256 256]


In [ ]:
slmobj_client.GetSLMTemperature()

## Send the example masks

In [5]:
slmobj_client.SetRefreshRate(10e-3)

height, width = slmobj_client.single_channel_shape

# phasemaskobj.LCOS_Clean()
check = bool(slmobj_client.WriteImageToSLM(np.zeros((height, width), dtype=np.uint8)))
check = bool(slmobj_client.WriteImageToSLM(np.zeros((height, width), dtype=np.uint8)))

# frame_NoStrip = Camobject.GetFrame()
MaskCMPLX = slm_masks.binary_stripe_phase(
    Nx=width,
    Ny=height,
    stripe_width=20,
    phase_value= + 0.0,
    orientation='vertical',
)

MaskInt = phasemaskobj.convert_phase_to_uint8(arr=MaskCMPLX[0, 0, :, :])
check = bool(slmobj_client.WriteImageToSLM(MaskInt))
check = bool(slmobj_client.WriteImageToSLM(MaskInt))

print(check)
# frame_Strip = Camobject.GetFrame()
# plt.subplot(1, 2, 1)
# plt.imshow(frame_NoStrip)
# plt.subplot(1, 2, 2)
# plt.imshow(frame_Strip)

True


In [7]:
phasemaskobj.Clear_Display()


## Cleanup

In [ ]:
# Run this when you are done with the notebook viewer/client.
# viewer.stopProcess()
# slmobj_client.release_control()
# slmobj_client.close()